# Tool Layer and Safe Tool Execution for a Parts-Pricing Agent

**Technical report 1 of the switchgear-estimation-assistant project**

| Item | Detail |
|---|---|
| Status | In progress |
| Covers | `tools.py` and `run_tool` in `agent.py`, both in `src/estimation_assistant/` |
| Later reports | Agent loop, output checks, retrieval with sources |
| Data | Synthetic only. No real catalogue, prices or drawings. |

## 1. Purpose and scope

Quoting electrical switchgear work means reading drawings and specifications, listing the parts, and adding up prices. The long-term aim of the project is an agent that does this and records every step it took.

This report covers the lowest layer: the functions an agent may call, and the code that runs them safely.

| In scope | Out of scope (later reports) |
|---|---|
| Three tools: price lookup, parts list, line total | The loop that talks to the model |
| A runner that executes a tool by name | Checking the final answer against a schema |
| Behaviour on bad input | Searching documents and citing sources |
| Manual tests | Reading drawing images |

## 2. Design principle

A language model produces text. It cannot run code. Tool calling therefore works in four steps.

```
Application                                   Model
    |--- question + list of tools ------------->|
    |<-- request: run line_total(95.0, 3) ------|
    |  (the application runs the function)      |
    |--- result, with the role "tool" --------->|
    |<-- final answer written from the result --|
```

Three consequences follow.

| Consequence | Meaning |
|---|---|
| Arithmetic is done by code | The model never calculates a price |
| Each number is traceable | A figure in an answer should match a tool call in the log |
| The application stays in control | Only functions on an approved list can run |

## 3. Environment

| Item | Detail |
|---|---|
| Model | Gemma 4, small size (`gemma4:e2b`), served by Ollama |
| Hardware | Google Colab, Tesla T4 GPU |
| Language | Python 3 with the Ollama Python client |

The tool layer does not call a model. All tests in this report use plain Python in a terminal.

## 4. Data

The catalogue is synthetic. Part names are generic industry terms. Prices are invented, in pounds, and exist only for testing.

| Part name | Meaning | Price (GBP, invented) |
|---|---|---|
| `mccb` | Moulded case circuit breaker | 145.00 |
| `acb` | Air circuit breaker | 2850.00 |
| `busbar` | Copper busbar, per metre | 95.00 |
| `current transformer` | Current transformer | 38.00 |
| `cable gland` | Cable gland | 25.00 |

**Data rules**

1. All data is invented.
2. No real prices, part numbers, drawings or customer details.
3. No text or images copied from any company catalogue.
4. No brand or product names.

## 5. Repository layout

```
src/estimation_assistant/
  tools.py    the catalogue and the three tools
  agent.py    runs tools safely and drives the agent loop
```

The two files change at different rates. New parts and tools change `tools.py`. The runner in `agent.py` changes rarely. Keeping them apart means adding a part never touches the runner.

## 6. The catalogue

```python
"""Tools for interacting with the product catalog (synthetic data)."""

PRICES = {
    "mccb": 145.00,
    "acb": 2850.00,
    "busbar": 95.00,
    "current transformer": 38.00,
    "cable gland": 25.00,
}
```

The file defines names only. It contains no `print` calls, so importing it produces no output.

## 7. Tool: `lookup_price`

```python
def lookup_price(part_name: str) -> str:
    """Return the unit price of one part, as text (for example 95.0).
    Valid part names: mccb (moulded case circuit breaker),
    acb (air circuit breaker), busbar (copper busbar, per metre),
    current transformer, cable gland.
    Use the exact name in lowercase.
    If the name is not valid, the reply says so and lists the valid part names.
    """
    #Normalize the part name before look up
    name = part_name.strip().lower()
    if name in PRICES:
        return str(PRICES[name])
    return f"unknown part '{part_name}'. Valid part: {', '.join(PRICES)}"
```

| Decision | Reason |
|---|---|
| The docstring lists every valid name | The model reads the docstring to decide how to call the tool. Without the names it has to guess. |
| The name is stripped and lowercased | `"  BusBar "` and `"busbar"` should both match |
| An unknown name returns a message, not an exception | The message names the bad input and lists valid ones, so the model can correct itself |
| The return type is `str` for both outcomes | Tool results travel to the model as text |

## 8. Tool: `list_parts`

```python
def list_parts() -> str:
    """List every part in the catalog, separated by commas."""
    return ", ".join(PRICES)
```

The tool takes no input. It reads the names from `PRICES`, so the list never falls out of date when a part is added.

## 9. Tool: `line_total`

```python
def line_total(unit_price: float, quantity: int) -> float:
    """Return the total price for one quote line, rounded to 2 decimal places.
    unit_price is the price of one item in pounds. Use the value from lookup_price.
    quantity is the number of items, as a whole number.
    """
    # Round to 2 decimals so the amount is a clean currency value
    return round(unit_price * quantity, 2)
```

Floating-point arithmetic can leave long tails. In plain Python, `0.1 * 3` gives `0.30000000000000004`. The call `round(..., 2)` removes the tail, so a quote shows a clean amount.

The sentence "Use the value from `lookup_price`" in the docstring tells the model that one tool feeds the next.

## 10. Verification of `tools.py`

These checks were run by hand before the restructure, from the old source folder. `tests/test_tools.py` now covers the tool results below.

| Test | Result |
|---|---|
| `lookup_price('busbar')` | `95.0` |
| `lookup_price('  Cable Gland ')` | `25.0` |
| `lookup_price('fuse')` | Message naming `fuse` and listing the five valid parts |
| `list_parts()` | `mccb, acb, busbar, current transformer, cable gland` |
| `line_total(95.0, 3)` | `285.0` |
| `line_total(1.75, 4)` | `7.0` |
| `line_total(0.1, 3)` | `0.3` |
| `0.1 * 3` (no rounding) | `0.30000000000000004` |
| `import tools` alone | No output |

## 11. Safe execution: `run_tool`

The runner turns a tool name (text) into the real function and calls it with the inputs the model supplied.

```python
from .tools import line_total, list_parts, lookup_price

TOOLS = {
    "lookup_price": lookup_price,
    "list_parts": list_parts,
    "line_total": line_total,
}


def run_tool(name: str, arguments: dict) -> str:
    """Run one tool by name and return its result as text.
    An unknown tool name, wrong inputs or any other error inside the tool return a message instead of raising an error."""
    # Reject names the model made up
    if name not in TOOLS:
        return f"Unknown tool '{name}'. Valid tools: {', '.join(TOOLS)}"
    try:
        return str(TOOLS[name](**arguments))
    except TypeError as error:
        # Raised when the inputs do not match the tool's parameters
        return f"Bad inputs for '{name}': {error}"
    except Exception as error:
        # Any other failure inside the tool is reported to the model instead of stopping the run
        return f"Tool '{name}' failed with {type(error).__name__}: {error}"
```

This report covers the first three tools. `search_notes` joined `TOOLS` later (report 3).

| Situation | Behaviour |
|---|---|
| Name is in `TOOLS` and inputs are valid | The result, as text |
| Name is not in `TOOLS` | A message listing the valid tool names |
| Inputs do not match the function | A message containing Python's own error text |
| Any other error inside the tool (added 3 Nov 2026) | A message naming the tool and the error type |

Since 3 Nov 2026, any other error inside a tool, such as a failed embedding call, also returns a message. Two pytest tests in `tests/test_agent.py` cover this.

**Why this matters.** Calling `TOOLS[name](**arguments)` directly would stop the whole agent on a made-up name or a missing input. Returning a message lets the model read the problem and try again.

## 12. Verification of `run_tool`

| Test | Result |
|---|---|
| `run_tool('lookup_price', {'part_name': 'busbar'})` | `95.0` |
| `run_tool('line_total', {'unit_price': 95.0, 'quantity': 3})` | `285.0` |
| `run_tool('list_parts', {})` | `mccb, acb, busbar, current transformer, cable gland` |
| `run_tool('make_coffee', {})` | `Unknown tool 'make_coffee'. Valid tools: lookup_price, list_parts, line_total` |
| `run_tool('line_total', {'unit_price': 95.0})` | `Bad inputs for 'line_total': line_total() missing 1 required positional argument: 'quantity'` |
| `run_tool('lookup_price', {'part_name': 5})` (3 Nov 2026) | `Tool 'lookup_price' failed with AttributeError: 'int' object has no attribute 'strip'` |
| `import agent` alone | No output |

(three tools at the time of this report; the code now lists four, including search_notes)

## 13. Observations from the prototype

An earlier notebook prototype used a three-part catalogue with different prices. It ran a loop of up to five rounds, and each executed tool printed one log line.

| Tool description | Question | Observed behaviour |
|---|---|---|
| Short, without the valid names | "What is the price of 3 busbars?" | No tool call. The model asked which type of busbar was meant. |
| Valid names listed in the description | "What is the price of one busbar?" | `Round 1: lookup_price({'part_name': 'busbar'}) -> 80.0`, then a final answer |
| Same | "How much do 3 busbars cost in total?" | `Round 1: lookup_price(...) -> 80.0`, then `Round 2: line_total({'quantity': 3, 'unit_price': 80}) -> 240` |
| Same, with a limit of one round | "How much do 3 busbars cost in total?" | `Round 1: lookup_price(...) -> 80.0`, then `Stopped: reached the round limit.` |

**Findings**

1. The tool description changed the behaviour. The loop code was identical in all runs.
2. The result of the first tool became the input of the second.
3. The round limit stopped the run cleanly and did not produce a guessed total.
4. Every number in a final answer matched a logged tool call. A number with no matching log line would indicate a guess by the model.

## 14. Limitations

| Limitation | Note |
|---|---|
| Small model | Facts it states from memory need checking. Prices come only from tools. |
| Text results | `lookup_price` returns text. In the prototype the model passed a numeric value to `line_total`, but this is not guaranteed. A schema check is planned. |
| One price per part type | Real quotes depend on ratings and sizes. The catalogue ignores them. |
| Test coverage | `tests/` holds pytest tests for `tools.py` and `checks.py`. `run_tool` lives in `agent.py` and has no automated test. |
| Synthetic data | Results say nothing about real pricing. |

## 15. Next steps

| Step | Content | Status |
|---|---|---|
| 1 | `run_agent`: the loop with a round limit and a stop rule | Done (report 2) |
| 2 | Schema checks on the final answer | Done (`schemas.py`) |
| 3 | Retrieval over a small synthetic catalogue, with the source kept in the run result | Done (report 3) |
| 4 | Reading drawing images | Not started |

## 16. Reproducing the tests

Run these from the repo root.

```powershell
$env:PYTHONPATH = "src"
python -c "from estimation_assistant import tools; print(tools.lookup_price('busbar'))"
python -c "from estimation_assistant import tools; print(tools.line_total(95.0, 3))"
python -c "from estimation_assistant import agent; print(agent.run_tool('make_coffee', {}))"
python -m pytest tests/test_tools.py
```

`tools.py` needs Python 3 only. Importing `agent.py` also needs the `ollama` package, and the tests need `pytest`.